In [2]:
import openai
import os
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.storage.index_store import SimpleIndexStore
from llama_index.core.storage.kvstore.simple_kvstore import SimpleKVStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import pprint
import faiss
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.vector_stores.faiss import FaissVectorStore
Settings.embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-large-v2") # MUST BE SAME AS THE ONE USED FOR INDEXING
embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-large-v2")

# Set Ollama as the default LLM globally
Settings.llm = Ollama(model="llama3.2", request_timeout=120, context_window=4096)
persist_dir = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\front_end\pipeline\data\Embedded"
faiss_path = "faiss.index"
faiss_file_path = os.path.join(persist_dir, faiss_path)

# Load existing FAISS and LlamaIndex
faiss_index = faiss.read_index(faiss_file_path)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(
    persist_dir=persist_dir,
    vector_store=vector_store
)
index = load_index_from_storage(storage_context)
# Sample query for testing
query_engine = index.as_query_engine(similarity_top_k=5)
response = query_engine.query("What is the orgnizational structure of Verztec?")
pprint.pp(response)

Response(response='The organizational structure of Verztec Consulting Pte Ltd '
                  'can be broken down into several key departments and '
                  'positions. At the top is the Chief Executive Officer (CEO), '
                  'who holds the highest authority.\n'
                  '\n'
                  'Below the CEO are two main reporting positions: the Chief '
                  'Operating Officer (COO) and the Senior Operations Manager. '
                  'The COO manages daily business operations, overseeing the '
                  'Admin & Operations and Finance departments. The Senior '
                  'Operations Manager reports to the COO and supervises the '
                  'Admin & Receptionist, as well as managing the Human '
                  'Resource department.\n'
                  '\n'
                  "The Finance department is also under the COO's supervision "
                  'and includes the Accounts team, which handles budgeting an

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

memory = ChatMemoryBuffer.from_defaults(token_limit=3000)

chat_engine = index.as_chat_engine(
    chat_mode="context",
    memory=memory,
    system_prompt=(
        "You are Verztec's friendly HR assistant, able to have normal interactions, as well as answer"
        "HR related queries. If the user asks a question in this format: **[Label]**: [Simplified question or concern]" \
        "you should respond with a detailed answer based on the information provided in the documents." \
        "If the user asks a question that is not related to HR, you should respond with a friendly message" \
    ),
    similarity_top_k=5
)

In [9]:

response = chat_engine.chat("Hello!")
print(response)


Hello! It's nice to meet you. Is there something I can help you with today? Perhaps you'd like some advice on writing an effective business email or need assistance with a specific HR-related query? I'm here to assist you!


In [14]:
raw_text = response.response.strip()
raw_text

"Hello! It's nice to meet you. Is there something I can help you with today? Perhaps you'd like some advice on writing an effective business email or need assistance with a specific HR-related query? I'm here to assist you!"

In [5]:

response = chat_engine.chat("It is 3am in Singapore now!")
print(response)


That's quite late! It's still early hours for most people, but I'm here to chat with you whenever you're awake. What brings you out and about at this hour? Don't worry, I won't judge – I'll just listen and try to help if I can!


In [6]:

response = chat_engine.stream_chat("What is verztecs organizational structure?")
for token in response.response_gen:
    print(token, end="")

According to the information provided earlier, Verztec Consulting Pte Ltd's organizational structure is as follows:

1. Chief Executive Officer (CEO) - The highest authority, overseeing all departments, operations, and strategic initiatives.
2. Chief Operating Officer (COO) - Reports directly to the CEO, managing daily business operations and supervising:
	* Admin & Operations
	* Finance departments
3. Senior Operations Manager - Reports to the COO, supervising:
	* Admin & Receptionist
	* Human Resource department
4. Service Delivery division - Reports directly to the CEO, including:
	* QA/QC team

Please note that this structure might be subject to change or updates, as organizational structures can evolve over time.

Also, I'd like to mention that Verztec Consulting Pte Ltd has a Quality Procedure in place, which includes monitoring customer satisfaction and implementing improvement actions. However, this is more of a operational procedure rather than an organizational structure per 

In [7]:

response = chat_engine.stream_chat("What was the first thing I told you ?")
for token in response.response_gen:
    print(token, end="")

You initially told me that it's 3am in Singapore, where we were having our conversation!